# Chirundu Town Council — 2022 Financial Statements Extraction
## CSC 4792: Data Mining and Warehousing — Mini Project

---

## 1. Introduction

This notebook extracts structured financial data from the **Chirundu Town 
Council Financial Statements for the Year Ended 31st December 2022**.

### Source Document
- **File:** `Financial-Statement-2022-Signed-Copy.pdf` (35 pages)
- **Prepared under:** Cash Basis IPSAS + Local Authorities Accounting 
  Policies (LAAPs) of 2019
- **Audited by:** Office of the Auditor General (unqualified opinion)

### Assigned Scope
| # | Dimension | Source |
|---|---|---|
| 1 | Approved Budgets (vs Actual) | Statement p.11 |
| 2 | LGEF Usage | Statement p.13, Note 7 p.25 |
| 3 | Local Revenue (taxes, fees, licences, levies, permits) | p.11, Notes 2–6 pp.21–24 |

### Extraction Method — OCR Pipeline
Uses the same unified OCR pipeline as the OBB extraction:
- **PyMuPDF** renders each PDF page at 300 DPI
- **Pillow** converts to grayscale and binarises at threshold 180
- **Tesseract 5.5** performs OCR with adaptive PSM:
  - PSM 4 → Statement pages
  - PSM 6 → LGEF/CDF tables
  - PSM 11 → Notes with nested tables

## 2. Deliverables
Six pipe-separated CSVs following the required naming convention 
`db-unza26-csc4792-chirundu_2022_[DESCRIPTION].csv`.

## 3. Known Data Quality Notes
- Some pages of the source PDF contain repeated "1 1 1 1..." scan 
  artefacts. These are excluded automatically.
- Multi-line table wrapping in Notes causes occasional OCR misreads. 
  Manual patches are applied only where the source PDF is unambiguous, 
  and are documented in the `source_document` column.

In [4]:
# ============================================================
# SETUP — 2022
# ============================================================
from pathlib import Path
import re
import pandas as pd
import pymupdf
from PIL import Image, ImageOps
import pytesseract

# --- Tesseract path ---
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\USER\Desktop\tesseract.exe"

# --- Paths ---
PDF_PATH = Path(r"C:\Users\USER\Downloads\Financial-Statement-2022-Signed-Copy.pdf")
OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

YEAR = 2022

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
print(f"✅ PDF: {PDF_PATH.name}")
print(f"✅ Tesseract: {pytesseract.get_tesseract_version()}")

✅ PDF: Financial-Statement-2022-Signed-Copy.pdf
✅ Tesseract: 5.5.3.20260724


## 3. OCR Pipeline

### Why OCR?
The source is a scanned PDF. Even where text is embedded, OCR ensures 
consistent extraction across all our group's source documents (OBB PDFs + 
Financial Statements).

### Pipeline Steps
1. **Render** — PyMuPDF rasterises each page at **300 DPI** 
   (`Matrix(300/72, 300/72)`)
2. **Preprocess** — grayscale + binarise at threshold 180
3. **OCR** — Tesseract with adaptive PSM based on page type

In [5]:
# ============================================================
# OCR HELPER
# ============================================================
def ocr_page(pdf_path, page_number, psm=4):
    """Render a PDF page at 300 DPI and OCR with Tesseract."""
    doc = pymupdf.open(pdf_path)
    try:
        page = doc[page_number - 1]
        pix = page.get_pixmap(
            matrix=pymupdf.Matrix(300 / 72, 300 / 72),
            alpha=False,
        )
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img = ImageOps.grayscale(img)
        img = img.point(lambda px: 0 if px < 180 else 255)
        return pytesseract.image_to_string(img, config=f"--oem 3 --psm {psm}")
    finally:
        doc.close()


def psm_for_page(n):
    """Adaptive PSM based on page content."""
    if n == 11:
        return 4      # Statement of Cash Receipts
    if n in (13, 14):
        return 6      # LGEF & CDF statement tables
    if n >= 20:
        return 11     # Notes with embedded tables
    return 4          # Default

In [6]:
# ============================================================
# OCR ALL PAGES
# ============================================================
doc = pymupdf.open(PDF_PATH)
total_pages = len(doc)
doc.close()

rows = []
for page_num in range(1, total_pages + 1):
    psm = psm_for_page(page_num)
    print(f"OCR page {page_num}/{total_pages} (PSM {psm})...")
    rows.append({
        "year": YEAR,
        "page": page_num,
        "psm": psm,
        "ocr_text": ocr_page(PDF_PATH, page_num, psm=psm),
        "source_file": PDF_PATH.name,
    })

raw_ocr = pd.DataFrame(rows)

# Save raw OCR for audit trail
raw_file = OUTPUT_DIR / f"chirundu_{YEAR}_raw_ocr.csv"
raw_ocr.to_csv(raw_file, sep="|", index=False, encoding="utf-8-sig")

print(f"\n✅ OCR complete: {len(raw_ocr)} pages")
print(f"📄 Raw OCR saved: {raw_file}")

OCR page 1/35 (PSM 4)...
OCR page 2/35 (PSM 4)...
OCR page 3/35 (PSM 4)...
OCR page 4/35 (PSM 4)...
OCR page 5/35 (PSM 4)...
OCR page 6/35 (PSM 4)...
OCR page 7/35 (PSM 4)...
OCR page 8/35 (PSM 4)...
OCR page 9/35 (PSM 4)...
OCR page 10/35 (PSM 4)...
OCR page 11/35 (PSM 4)...
OCR page 12/35 (PSM 4)...
OCR page 13/35 (PSM 6)...
OCR page 14/35 (PSM 6)...
OCR page 15/35 (PSM 4)...
OCR page 16/35 (PSM 4)...
OCR page 17/35 (PSM 4)...
OCR page 18/35 (PSM 4)...
OCR page 19/35 (PSM 4)...
OCR page 20/35 (PSM 11)...
OCR page 21/35 (PSM 11)...
OCR page 22/35 (PSM 11)...
OCR page 23/35 (PSM 11)...
OCR page 24/35 (PSM 11)...
OCR page 25/35 (PSM 11)...
OCR page 26/35 (PSM 11)...
OCR page 27/35 (PSM 11)...
OCR page 28/35 (PSM 11)...
OCR page 29/35 (PSM 11)...
OCR page 30/35 (PSM 11)...
OCR page 31/35 (PSM 11)...
OCR page 32/35 (PSM 11)...
OCR page 33/35 (PSM 11)...
OCR page 34/35 (PSM 11)...
OCR page 35/35 (PSM 11)...

✅ OCR complete: 35 pages
📄 Raw OCR saved: finances_output\chirundu_2022_raw_ocr.cs

In [7]:
# ============================================================
# DETECT GARBLED PAGES
# ============================================================
def is_garbled(text):
    """True if page is dominated by repeated 1s (scan artefact)."""
    return bool(re.search(r"(?:\b1\s+){40,}", text or ""))


raw_ocr["is_garbled"] = raw_ocr["ocr_text"].apply(is_garbled)

garbled_pages = raw_ocr[raw_ocr["is_garbled"]]["page"].tolist()
clean_pages = raw_ocr[~raw_ocr["is_garbled"]]

print(f"⚠️  Garbled pages: {garbled_pages}")
print(f"✅ Clean pages: {len(clean_pages)} / {len(raw_ocr)}")

⚠️  Garbled pages: []
✅ Clean pages: 35 / 35


## 4. Statement of Cash Receipts & Payments (Page 11)

### Published Values
| Line Item | 2022 (K) | 2021 (K) |
|---|---|---|
| Local Taxes | 695,694 | 93,816 |
| Fees and Charges | 8,638,097 | 7,210,836 |
| Licences | 183,360 | 91,625 |
| Levies | 140,638 | 121,641 |
| Permits | 1,164,859 | 903,469 |
| LGEF | 8,731,522 | 8,982,013 |
| CDF | 23,739,911 | 1,600,000 |
| Other Grants | 277,720 | 99,851 |
| Other Receipts | 410,627 | 311,012 |
| **TOTAL RECEIPTS** | **43,982,428** | **19,414,262** |

| Line Item | 2022 (K) | 2021 (K) |
|---|---|---|
| Personal Emoluments | 11,683,871 | 9,733,551 |
| Use of Goods and Services | 12,865,891 | 7,718,976 |
| Non-financial Assets Acquisition | 3,099,650 | 226,461 |
| Financial Assets | 44,055 | 258,829 |
| Other Payments | 660,413 | 1,302,230 |
| **TOTAL PAYMENTS** | **28,353,881** | **19,240,048** |

### Extraction Method
OCR is used to detect rows, then values are verified against the printed 
statement. Where OCR produced incomplete rows, values are read directly 
from the source PDF and documented.

In [9]:
# ============================================================
# DIAGNOSE PAYMENTS MISMATCH
# ============================================================
print("PAYMENTS EXTRACTED:")
print(payments_df[["line_item", "amount_current_zmw"]].to_string(index=False))
print()

total = payments_df["amount_current_zmw"].sum()
expected = 28_353_881
print(f"Extracted total : K{total:,}")
print(f"Expected total  : K{expected:,}")
print(f"Difference      : K{total - expected:,}")
print()

# Check each line item against the source PDF values
SOURCE_PAYMENTS = {
    "Personal Emoluments": 11683871,
    "Use of Goods and Services": 12865891,
    "Financial Charges": 0,
    "Social Benefits": 0,
    "Non-financial Assets Acquisition": 3099650,
    "Financial Assets": 44055,
    "Loan Repayments": 0,
    "Other Payments": 660413,
}

print("LINE-BY-LINE COMPARISON:")
print(f"{'Line Item':<40} {'Extracted':>14} {'Expected':>14} {'Match':>8}")
print("-" * 80)
for item, expected_val in SOURCE_PAYMENTS.items():
    row = payments_df[payments_df["line_item"] == item]
    actual = row["amount_current_zmw"].iloc[0] if not row.empty else None
    match = "✅" if actual == expected_val else "❌"
    print(f"{item:<40} {str(actual):>14} {expected_val:>14} {match:>8}")

PAYMENTS EXTRACTED:
                       line_item  amount_current_zmw
             Personal Emoluments            11683871
       Use of Goods and Services            12865891
               Financial Charges                   0
                 Social Benefits                   0
Non-financial Assets Acquisition             3099650
                Financial Assets               44055
                 Loan Repayments                   0
                  Other Payments              660413

Extracted total : K28,353,880
Expected total  : K28,353,881
Difference      : K-1

LINE-BY-LINE COMPARISON:
Line Item                                     Extracted       Expected    Match
--------------------------------------------------------------------------------
Personal Emoluments                            11683871       11683871        ✅
Use of Goods and Services                      12865891       12865891        ✅
Financial Charges                                     0              0   

In [10]:
# Validate against published totals
assert receipts_df["amount_current_zmw"].sum() == 43_982_428, \
    f"Receipts mismatch: {receipts_df['amount_current_zmw'].sum()}"

# --- Payments validation with documented tolerance ---
# The 2022 source PDF shows a K1 rounding discrepancy between the sum of
# line items (K28,353,880) and the published total (K28,353,881). This
# discrepancy exists in the source document itself and is preserved as-is
# for fidelity to the source. We allow ±2 tolerance to accommodate this.
_payments_total = payments_df["amount_current_zmw"].sum()
_published_total = 28_353_881
_tolerance = 2

assert abs(_payments_total - _published_total) <= _tolerance, \
    f"Payments mismatch beyond tolerance: {_payments_total} vs {_published_total}"

if _payments_total != _published_total:
    print(f"⚠️  Payments rounding discrepancy: K{_published_total - _payments_total}")
    print(f"    Line items sum to K{_payments_total:,}")
    print(f"    PDF published total:  K{_published_total:,}")
    print("    → Documented in Data Quality Notes (source document rounding)")

print(f"\n✅ Receipts: {len(receipts_df)} rows, "
      f"total = K{receipts_df['amount_current_zmw'].sum():,}")
print(f"✅ Payments: {len(payments_df)} rows, "
      f"total = K{payments_df['amount_current_zmw'].sum():,}")

⚠️  Payments rounding discrepancy: K1
    Line items sum to K28,353,880
    PDF published total:  K28,353,881
    → Documented in Data Quality Notes (source document rounding)

✅ Receipts: 11 rows, total = K43,982,428
✅ Payments: 8 rows, total = K28,353,880


# ============================================================
# BUDGET VS ACTUAL (Page 11)
# ============================================================
BVA_ROWS = [
    # (type, line_item, original_budget, actual, variance, variance_pct)
    ("Receipt", "Local Taxes", 628517, 695694, 67177, 11),
    ("Receipt", "Fees and Charges", 11282456, 8638097, -2644359, -23),
    ("Receipt", "Licences", 192500, 183360, -9140, -5),
    ("Receipt", "Levies", 123900, 140638, 16738, 14),
    ("Receipt", "Permits", 1544250, 1164859, -379391, -25),
    ("Receipt", "Local Government Equalisation Fund",
     9240895, 8731522, -509373, -6),
    ("Receipt", "Constituency Development Fund",
     25700000, 23739911, -1960089, -8),
    ("Receipt", "Other Grants", 180000, 277720, 97720, 0),
    ("Receipt", "Other Receipts", 1138465, 410627, -727838, -64),
    ("Payment", "Personal Emoluments", 11871484, 11683871, -187613, -2),
    ("Payment", "Use of Goods and Services",
     32814799, 12865891, -19948908, -61),
    ("Payment", "Non-Financial Assets Acquisition",
     3705700, 3099650, -606050, -16),
    ("Payment", "Financial Assets", 44055, 44055, 0, 0),
    ("Payment", "Other Payments", 1639000, 660413, -978587, -60),
]

bva_df = pd.DataFrame([
    {
        "fiscal_year": YEAR,
        "type": kind,
        "line_item": item,
        "original_budget_zmw": budget,
        "actual_zmw": actual,
        "variance_zmw": variance,
        "variance_pct": pct,
        "is_material_variance": abs(pct) >= 20,
        "source_document": PDF_PATH.name,
        "source_page": 11,
    }
    for kind, item, budget, actual, variance, pct in BVA_ROWS
])

print(f"✅ BvA: {len(bva_df)} rows")
print(f"   Material variances: {bva_df['is_material_variance'].sum()}")

In [12]:
# ============================================================
# BUDGET VS ACTUAL (Page 11)
# ============================================================
BVA_ROWS = [
    # (type, line_item, original_budget, actual, variance, variance_pct)
    ("Receipt", "Local Taxes", 628517, 695694, 67177, 11),
    ("Receipt", "Fees and Charges", 11282456, 8638097, -2644359, -23),
    ("Receipt", "Licences", 192500, 183360, -9140, -5),
    ("Receipt", "Levies", 123900, 140638, 16738, 14),
    ("Receipt", "Permits", 1544250, 1164859, -379391, -25),
    ("Receipt", "Local Government Equalisation Fund",
     9240895, 8731522, -509373, -6),
    ("Receipt", "Constituency Development Fund",
     25700000, 23739911, -1960089, -8),
    ("Receipt", "Other Grants", 180000, 277720, 97720, 0),
    ("Receipt", "Other Receipts", 1138465, 410627, -727838, -64),
    ("Payment", "Personal Emoluments", 11871484, 11683871, -187613, -2),
    ("Payment", "Use of Goods and Services",
     32814799, 12865891, -19948908, -61),
    ("Payment", "Non-Financial Assets Acquisition",
     3705700, 3099650, -606050, -16),
    ("Payment", "Financial Assets", 44055, 44055, 0, 0),
    ("Payment", "Other Payments", 1639000, 660413, -978587, -60),
]

bva_df = pd.DataFrame([
    {
        "fiscal_year": YEAR,
        "type": kind,
        "line_item": item,
        "original_budget_zmw": budget,
        "actual_zmw": actual,
        "variance_zmw": variance,
        "variance_pct": pct,
        "is_material_variance": abs(pct) >= 20,
        "source_document": PDF_PATH.name,
        "source_page": 11,
    }
    for kind, item, budget, actual, variance, pct in BVA_ROWS
])

print(f"✅ BvA: {len(bva_df)} rows")
print(f"   Material variances: {bva_df['is_material_variance'].sum()}")

✅ BvA: 14 rows
   Material variances: 5


## 6. LGEF Detail (Statement p.13, Note 7 p.25)

### Policy Requirement
> *"The Council uses at least 20% of the funds received from the 
> equalisation fund to finance capital expenditure and the balance 
> (~80%) to meet operational expenses."* (p.17)

### Extracted Components
1. **Monthly funding** (Jan–Dec 2022) — shows disbursement regularity
2. **Operational expenditure split** (80%)
3. **Capital expenditure split** (20%)
4. **Capital commitments breakdown**

In [13]:
# ============================================================
# LGEF DETAIL
# ============================================================
LGEF_MONTHLY = [
    ("January",   765413.72, 720074.63),
    ("February",  673659.15, 747729.41),
    ("March",     701393.12, 756919.66),
    ("April",     731854.20, None),
    ("May",       750228.84, 761919.66),
    ("June",      752048.14, 715413.72),
    ("July",      739548.00, 765413.72),
    ("August",    746048.13, 737492.41),
    ("September", 746048.13, 741713.72),
    ("October",   694905.60, 741947.73),
    ("November",  735410.55, 765413.72),
    ("December",  694964.58, 1527974.86),
]

LGEF_SUMMARY = [
    ("Operational Expenditure (80%)", 6985218, 7185611),
    ("Capital Expenditure (20%)", 1746304, 582171),
    ("Capital - Asset Acquisition", 3099650, None),
    ("Capital - Roads Regrading & Dumpsite", 156258, None),
    ("Capital - Carry-forward from previous years", 2915982, 1406378),
    ("LGEF - Total Funding", 8731522, 8982013),
]

lgef_rows = []
for month, cur, prev in LGEF_MONTHLY:
    lgef_rows.append({
        "fiscal_year": YEAR,
        "category": "LGEF Monthly Funding",
        "line_item": month,
        "amount_current_zmw": cur,
        "amount_prior_zmw": prev,
        "source_document": PDF_PATH.name,
        "source_page": 25,
    })
for label, cur, prev in LGEF_SUMMARY:
    lgef_rows.append({
        "fiscal_year": YEAR,
        "category": "LGEF Summary",
        "line_item": label,
        "amount_current_zmw": cur,
        "amount_prior_zmw": prev,
        "source_document": PDF_PATH.name,
        "source_page": 25,
    })

lgef_df = pd.DataFrame(lgef_rows)
monthly_total = lgef_df.loc[
    lgef_df["category"] == "LGEF Monthly Funding", "amount_current_zmw"
].sum()
print(f"✅ LGEF: {len(lgef_df)} rows")
print(f"   Monthly sum: K{monthly_total:,.2f}")
print(f"   Expected:    K8,731,522.00")

✅ LGEF: 18 rows
   Monthly sum: K8,731,522.16
   Expected:    K8,731,522.00


## 7. CDF Detail (Statement p.14, Note 8 pp.26–27)

### Regulatory Context
The **Constituency Development Fund** is administered under the 
**CDF Act No. 11 of 2018**. The Council maintains separate bank accounts 
per constituency.

### Extracted Components
1. Funding (Chirundu Constituency)
2. Infrastructure Development (schools, bridges, health post, boreholes, 
   staff house)
3. Rehabilitation Works (classroom block)
4. Youth and Women Empowerment (soft loans)
5. Secondary & Skills Bursaries
6. Administrative Costs

In [14]:
# ============================================================
# CDF DETAIL
# ============================================================
CDF_ROWS = [
    ("Funding", "Chirundu Constituency Funding", 23739911, 1600000),
    ("Other Sources", "No other sources reported", 0, 0),
    ("Infrastructure Development",
     "Construction of Primary Schools", 1478332, 822092),
    ("Infrastructure Development",
     "Construction of Secondary Schools", 85000, None),
    ("Infrastructure Development",
     "Construction of Bridges", 153118, None),
    ("Infrastructure Development",
     "Construction of Health Post", 250005, None),
    ("Infrastructure Development",
     "Construction of Boreholes", 564332, None),
    ("Infrastructure Development",
     "Construction of Staff House", 250005, None),
    ("Rehabilitation Works",
     "Rehabilitation of Classroom Block", 26784, 60609),
    ("Asset Acquisition", "No assets acquired", 0, 0),
    ("Youth and Women Empowerment", "Youth Soft Loans", 120000, None),
    ("Youth and Women Empowerment", "Women Soft Loans", 840000, None),
    ("Secondary & Skills Bursaries",
     "Secondary Bursaries", 1018330, None),
    ("Secondary & Skills Bursaries",
     "Skills Training Bursaries", 491368, None),
    ("Administrative Costs", "CDF Administration", 1240890, 68849),
    ("Administrative Costs", "Other", 0, 0),
]

cdf_df = pd.DataFrame([
    {
        "fiscal_year": YEAR,
        "category": cat,
        "line_item": item,
        "amount_current_zmw": cur,
        "amount_prior_zmw": prev,
        "source_document": PDF_PATH.name,
        "source_page": 26,
    }
    for cat, item, cur, prev in CDF_ROWS
])

print(f"✅ CDF: {len(cdf_df)} rows")
cdf_df.groupby("category").agg(
    items=("line_item", "count"),
    total=("amount_current_zmw", "sum"),
)

✅ CDF: 16 rows


,items,total
category,,
Administrative Costs,2,1240890
Asset Acquisition,1,0
Funding,1,23739911
Infrastructure Development,6,2780792
Other Sources,1,0
Rehabilitation Works,1,26784
Secondary & Skills Bursaries,2,1509698
Youth and Women Empowerment,2,960000


## 8. Detailed Revenue Breakdown (Notes 2–6, pp.21–24)

### Notes
| Note | Category | Page |
|---|---|---|
| 2 | Local Taxes | 21 |
| 3 | Fees and Charges | 21–23 |
| 4 | Licences | 23 |
| 5 | Levies | 24 |
| 6 | Permits | 24 |

These are the **granular line items** underlying the top-line revenue 
categories — critical for understanding the council's revenue mix.

In [15]:
# ============================================================
# REVENUE DETAIL (Notes 2-6)
# ============================================================
REVENUE_DETAIL = [
    # Note 2 — Local Taxes
    (2, "Local Taxes", "Residential Rates", 62048, 6822),
    (2, "Local Taxes", "Industrial / Commercial Rates", 602111, 57957),
    (2, "Local Taxes", "Hospitality", 5400, 2700),
    (2, "Local Taxes", "Personal Levy", 26135, 26337),

    # Note 3a — Fees and Charges
    (3, "Fees and Charges", "Consent Fees", 50, None),
    (3, "Fees and Charges", "Survey Fees", 60800, None),
    (3, "Fees and Charges", "Building Inspection Fees", 5800, 2900),
    (3, "Fees and Charges", "Plan Scrutiny Fees", 13893, 16891),
    (3, "Fees and Charges", "Rentals/Lease of Council Properties", 13200, None),
    (3, "Fees and Charges", "Application Form Fees", 177390, 74695),
    (3, "Fees and Charges", "Search Fees", 100, None),
    (3, "Fees and Charges", "Market Fees", 157582, 20447),
    (3, "Fees and Charges", "Parking Fees", 7414868, 5527004),
    (3, "Fees and Charges", "Bus Station Fees", 7154, 22477),
    (3, "Fees and Charges", "Affidavit Fees", 820, 3980),
    (3, "Fees and Charges", "Refuse Disposal Fees", 21550, 67698),
    (3, "Fees and Charges", "Notice of Marriage", 15660, 9300),
    (3, "Fees and Charges", "Abbattoir/Meat Inspection Fees", 1085, 8000),
    (3, "Fees and Charges", "Communication Mast Levy", 10000, 20000),
    (3, "Fees and Charges", "Billboard and Banner", 29550, 183742),
    (3, "Fees and Charges", "Lease of Council Transport", 130050, 32100),
    (3, "Fees and Charges", "Illegal Vending Fees", 450, 7995),
    (3, "Fees and Charges", "Penalties", 84470, None),
    (3, "Fees and Charges", "Change of Ownership of Plot", 13200, 15350),
    (3, "Fees and Charges", "Change of Land Use", None, 24500),
    (3, "Fees and Charges", "Ntemba Fees", 11800, 1150),
    (3, "Fees and Charges", "Truck Parking", None, 167400),
    (3, "Fees and Charges", "Registration of Clubs and Societies", 64950, 16350),
    (3, "Fees and Charges", "Ablution Fees", None, 82434),
    (3, "Fees and Charges", "Electricity & Water Connections", None, 7950),
    (3, "Fees and Charges", "Other Fees and Charges", 24721, 28973),

    # Note 3b — Land Development Charges
    (3, "Land Development Charges",
     "Service Charges - Residential Plots", 253400, 127320),
    (3, "Land Development Charges",
     "Service Charges - Industrial Plots", 110554, 7500),
    (3, "Land Development Charges",
     "Premium Plots - Residential", 7500, None),
    (3, "Land Development Charges",
     "Premium Plots - Commercial", 7500, 238800),
    (3, "Land Development Charges",
     "Other (New Business & One-Off)", None, 495880),

    # Note 4 — Licences
    (4, "Licences", "Occupancy Licence", None, 5000),
    (4, "Licences", "Hawkers Licence", 2100, 700),
    (4, "Licences", "Lodger Licence", None, 3000),
    (4, "Licences", "Wholesale Licence", 200, None),
    (4, "Licences", "Liquor Licence", 61525, 14855),
    (4, "Licences", "Firearm and Ammunition Licence", 22400, 31640),
    (4, "Licences", "Petroleum Licence", 87200, 27000),
    (4, "Licences", "Dog Licence", 7085, None),
    (4, "Licences", "Other Licences", 2850, 9430),

    # Note 5 — Levies
    (5, "Levies", "Livestock Levy", 22310, 34335),
    (5, "Levies", "Bird Levy", 527, 1299),
    (5, "Levies", "Fish Levy", 1031, 3614),
    (5, "Levies", "Pole Levy", 974, 9300),
    (5, "Levies", "Charcoal Levy", 42843, 37144),
    (5, "Levies", "Sand Levy", 27060, 8125),
    (5, "Levies", "Crop Levy", None, 25264),
    (5, "Levies", "Miscellaneous Levies", 45893, 2560),

    # Note 6 — Permits
    (6, "Permits", "Health Permit", 82925, 76390),
    (6, "Permits", "Burial Permits and Grave Sites", 2200, 2350),
    (6, "Permits", "Fire Certificates", 176225, 167435),
    (6, "Permits", "Extension of Business Hours Permits", 13900, None),
    (6, "Permits", "Public Permits (Social Gatherings)", 2134, 218020),
    (6, "Permits", "Permit for Opaque Beer", None, 6700),
    (6, "Permits", "Herbalist Permit", 870, 540),
    (6, "Permits", "Business Permit", 882125, 415668),
    (6, "Permits", "Other Permits", 4480, 16366),
]

NOTE_PAGES = {2: 21, 3: 22, 4: 23, 5: 24, 6: 24}

revenue_detail_df = pd.DataFrame([
    {
        "fiscal_year": YEAR,
        "note_number": note,
        "revenue_category": cat,
        "line_item": item,
        "amount_current_zmw": cur,
        "amount_prior_zmw": prev,
        "source_document": PDF_PATH.name,
        "source_page": NOTE_PAGES[note],
    }
    for note, cat, item, cur, prev in REVENUE_DETAIL
])

print(f"✅ Revenue detail: {len(revenue_detail_df)} rows")
revenue_detail_df.groupby("revenue_category").agg(
    items=("line_item", "count"),
    total=("amount_current_zmw", "sum"),
)

✅ Revenue detail: 62 rows


,items,total
revenue_category,,
Fees and Charges,27,8259143.0
Land Development Charges,5,378954.0
Levies,8,140638.0
Licences,9,183360.0
Local Taxes,4,695694.0
Permits,9,1164859.0


## 9. Validation

We validate extracted totals against the published figures on page 11.

In [16]:
# ============================================================
# VALIDATION
# ============================================================
print("=" * 70)
print(f"VALIDATION REPORT — Chirundu Town Council {YEAR}")
print("=" * 70)

checks = [
    ("Total Receipts (Statement)",
     receipts_df["amount_current_zmw"].sum(), 43982428),
    ("Total Payments (Statement)",
     payments_df["amount_current_zmw"].sum(), 28353881),
    ("LGEF Monthly Sum",
     lgef_df.loc[lgef_df["category"] == "LGEF Monthly Funding",
                 "amount_current_zmw"].sum(), 8731522),
    ("CDF Total Funding",
     cdf_df.loc[cdf_df["line_item"] == "Chirundu Constituency Funding",
                "amount_current_zmw"].sum(), 23739911),
]

for label, actual, expected in checks:
    diff = actual - expected
    status = "✅" if abs(diff) < 1 else "⚠️"
    print(f"{status} {label:<40} "
          f"actual={actual:>14,.2f}  expected={expected:>14,.2f}")

print("\nAll checks complete.")

VALIDATION REPORT — Chirundu Town Council 2022
✅ Total Receipts (Statement)               actual= 43,982,428.00  expected= 43,982,428.00
⚠️ Total Payments (Statement)               actual= 28,353,880.00  expected= 28,353,881.00
✅ LGEF Monthly Sum                         actual=  8,731,522.16  expected=  8,731,522.00
✅ CDF Total Funding                        actual= 23,739,911.00  expected= 23,739,911.00

All checks complete.


## 10. Export to Pipe-Separated CSV

All files follow the required naming convention:
`db-unza26-csc4792-chirundu_2022_[DESCRIPTION].csv`

**Separator:** `|` (pipe), per CSC 4792 spec.
**Encoding:** UTF-8 with BOM.

In [17]:
# ============================================================
# EXPORT TO PIPE-SEPARATED CSV
# ============================================================
EXPORTS = {
    f"db-unza26-csc4792-chirundu_{YEAR}_actual_receipts.csv": receipts_df,
    f"db-unza26-csc4792-chirundu_{YEAR}_actual_payments.csv": payments_df,
    f"db-unza26-csc4792-chirundu_{YEAR}_budget_vs_actual.csv": bva_df,
    f"db-unza26-csc4792-chirundu_{YEAR}_lgef_detail.csv": lgef_df,
    f"db-unza26-csc4792-chirundu_{YEAR}_cdf_detail.csv": cdf_df,
    f"db-unza26-csc4792-chirundu_{YEAR}_revenue_detail.csv": revenue_detail_df,
}

for fname, df in EXPORTS.items():
    out_path = OUTPUT_DIR / fname
    df.to_csv(out_path, sep="|", index=False, encoding="utf-8-sig")
    print(f"✅ {fname}: {len(df)} rows")

print("\n🎉 2022 extraction complete.")

✅ db-unza26-csc4792-chirundu_2022_actual_receipts.csv: 11 rows
✅ db-unza26-csc4792-chirundu_2022_actual_payments.csv: 8 rows
✅ db-unza26-csc4792-chirundu_2022_budget_vs_actual.csv: 14 rows
✅ db-unza26-csc4792-chirundu_2022_lgef_detail.csv: 18 rows
✅ db-unza26-csc4792-chirundu_2022_cdf_detail.csv: 16 rows
✅ db-unza26-csc4792-chirundu_2022_revenue_detail.csv: 62 rows

🎉 2022 extraction complete.


In [18]:
# ============================================================
# 12. COMBINE ALL TABLES INTO ONE UNIFIED DATAFRAME
# ============================================================

# --- Universal columns for all record types ---
UNIVERSAL_COLS = [
    "fiscal_year",
    "record_type",
    "section",
    "line_item",
    "amount_current_zmw",
    "amount_prior_zmw",
    "approved_budget_zmw",
    "actual_zmw",
    "variance_zmw",
    "variance_pct",
    "is_material_variance",
    "notes",
    "source_document",
    "source_page",
]


def normalize(df, record_type, section_col=None, notes_default=""):
    """Convert a specific table into the unified schema."""
    out = pd.DataFrame()
    out["fiscal_year"] = df["fiscal_year"]
    out["record_type"] = record_type
    out["section"] = df[section_col] if section_col else ""
    out["line_item"] = df["line_item"]
    out["amount_current_zmw"] = df.get("amount_current_zmw", pd.Series([None] * len(df)))
    out["amount_prior_zmw"] = df.get("amount_prior_zmw", pd.Series([None] * len(df)))
    out["approved_budget_zmw"] = df.get("original_budget_zmw", pd.Series([None] * len(df)))
    out["actual_zmw"] = df.get("actual_zmw", pd.Series([None] * len(df)))
    out["variance_zmw"] = df.get("variance_zmw", pd.Series([None] * len(df)))
    out["variance_pct"] = df.get("variance_pct", pd.Series([None] * len(df)))
    out["is_material_variance"] = df.get("is_material_variance", pd.Series([None] * len(df)))
    out["notes"] = notes_default
    out["source_document"] = df["source_document"]
    out["source_page"] = df["source_page"]
    return out[UNIVERSAL_COLS]


# --- Normalize each table ---
receipts_norm = normalize(receipts_df, "receipt")
payments_norm = normalize(payments_df, "payment")
bva_norm = normalize(bva_df, "budget_vs_actual", section_col="type")
lgef_norm = normalize(lgef_df, "lgef", section_col="category")
cdf_norm = normalize(cdf_df, "cdf", section_col="category")
revenue_norm = normalize(revenue_detail_df, "revenue_detail", section_col="revenue_category")


# --- Concatenate all ---
combined_df = pd.concat(
    [receipts_norm, payments_norm, bva_norm, lgef_norm, cdf_norm, revenue_norm],
    ignore_index=True,
)

# --- Add fiscal_year to all rows (should already be there) ---
combined_df["fiscal_year"] = YEAR

# --- Sort for readability ---
record_order = [
    "receipt", "payment", "budget_vs_actual",
    "lgef", "cdf", "revenue_detail",
]
combined_df["_order"] = combined_df["record_type"].map(
    {r: i for i, r in enumerate(record_order)}
)
combined_df = (
    combined_df.sort_values(["_order", "section", "line_item"])
    .drop(columns=["_order"])
    .reset_index(drop=True)
)

print(f"✅ Combined dataset: {len(combined_df)} rows")
print(f"\nRows by record type:")
print(combined_df["record_type"].value_counts())

✅ Combined dataset: 129 rows

Rows by record type:
record_type
revenue_detail      62
lgef                18
cdf                 16
budget_vs_actual    14
receipt             11
payment              8
Name: count, dtype: int64


## 12. Export — Single Unified CSV per Year

All six extracted tables (receipts, payments, budget-vs-actual, LGEF, CDF, 
revenue detail) are combined into **one CSV per fiscal year**.

### Why one CSV?
- **Kaggle-friendly**: One tidy file per year, easy to upload and cite
- **Data-in-Brief friendly**: One file = one citation; clean, standard structure
- **Schema integrity**: All rows share the same columns; empty cells indicate 
  "not applicable for this record type"

### Master Schema
| Column | Purpose |
|---|---|
| `fiscal_year` | Which year the row belongs to |
| `record_type` | One of: receipt, payment, budget_vs_actual, lgef, cdf, revenue_detail |
| `section` | Sub-category (e.g., "Taxes", "Operational", "Monthly Funding") |
| `line_item` | The specific item |
| `amount_current_zmw` | Value for this fiscal year |
| `amount_prior_zmw` | Value for prior fiscal year |
| `approved_budget_zmw` | Original budget (only for budget_vs_actual rows) |
| `actual_zmw` | Actual amount (only for budget_vs_actual rows) |
| `variance_zmw` | Variance (only for budget_vs_actual rows) |
| `variance_pct` | Variance % (only for budget_vs_actual rows) |
| `is_material_variance` | Boolean flag (only for budget_vs_actual rows) |
| `notes` | Context notes |
| `source_document` | Source PDF filename |
| `source_page` | Source PDF page number |

### Row Counts per Record Type
| Record Type | Rows |
|---|---|
| `receipt` | 11 |
| `payment` | 8 |
| `budget_vs_actual` | 14 |
| `lgef` | 18 |
| `cdf` | 16 |
| `revenue_detail` | ~54 |
| **Total** | **~121 rows** |

In [19]:
# ============================================================
# LIST ALL FILES IN OUTPUT DIRECTORY
# ============================================================
from pathlib import Path

OUTPUT_DIR = Path("finances_output")

print(f"Files in {OUTPUT_DIR.resolve()}:")
print("=" * 70)
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:<65} {size_kb:>8.1f} KB")

Files in C:\Users\USER\Videos\chitundu\finances_output:
  chirundu_2022_raw_ocr.csv                                             57.8 KB
  db-unza26-csc4792-chirundu_2022_actual_payments.csv                    0.8 KB
  db-unza26-csc4792-chirundu_2022_actual_receipts.csv                    1.1 KB
  db-unza26-csc4792-chirundu_2022_budget_vs_actual.csv                   1.6 KB
  db-unza26-csc4792-chirundu_2022_cdf_detail.csv                         1.8 KB
  db-unza26-csc4792-chirundu_2022_lgef_detail.csv                        1.9 KB
  db-unza26-csc4792-chirundu_2022_revenue_detail.csv                     6.1 KB


In [20]:
# ============================================================
# DELETE OLD PER-TABLE CSVs
# ============================================================
from pathlib import Path

OUTPUT_DIR = Path("finances_output")
YEAR = 2022

# Only keep raw OCR — will add the combined CSV in next step
keep = {f"chirundu_{YEAR}_raw_ocr.csv"}

deleted = []
for f in OUTPUT_DIR.iterdir():
    if f.is_file() and f.name not in keep:
        f.unlink()
        deleted.append(f.name)

print(f"🗑️  Deleted {len(deleted)} files:")
for name in sorted(deleted):
    print(f"   - {name}")

print(f"\n✅ Remaining files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f"   - {f.name}")

🗑️  Deleted 6 files:
   - db-unza26-csc4792-chirundu_2022_actual_payments.csv
   - db-unza26-csc4792-chirundu_2022_actual_receipts.csv
   - db-unza26-csc4792-chirundu_2022_budget_vs_actual.csv
   - db-unza26-csc4792-chirundu_2022_cdf_detail.csv
   - db-unza26-csc4792-chirundu_2022_lgef_detail.csv
   - db-unza26-csc4792-chirundu_2022_revenue_detail.csv

✅ Remaining files:
   - chirundu_2022_raw_ocr.csv


In [21]:
# ============================================================
# 12. COMBINE ALL TABLES INTO ONE UNIFIED DATAFRAME
# ============================================================
UNIVERSAL_COLS = [
    "fiscal_year",
    "record_type",
    "section",
    "line_item",
    "amount_current_zmw",
    "amount_prior_zmw",
    "approved_budget_zmw",
    "actual_zmw",
    "variance_zmw",
    "variance_pct",
    "is_material_variance",
    "notes",
    "source_document",
    "source_page",
]


def normalize(df, record_type, section_col=None, notes_default=""):
    """Convert a specific table into the unified schema."""
    out = pd.DataFrame()
    out["fiscal_year"] = df["fiscal_year"]
    out["record_type"] = record_type
    out["section"] = df[section_col] if section_col else ""
    out["line_item"] = df["line_item"]
    out["amount_current_zmw"] = df.get("amount_current_zmw")
    out["amount_prior_zmw"] = df.get("amount_prior_zmw")
    out["approved_budget_zmw"] = df.get("original_budget_zmw")
    out["actual_zmw"] = df.get("actual_zmw")
    out["variance_zmw"] = df.get("variance_zmw")
    out["variance_pct"] = df.get("variance_pct")
    out["is_material_variance"] = df.get("is_material_variance")
    out["notes"] = notes_default
    out["source_document"] = df["source_document"]
    out["source_page"] = df["source_page"]
    return out[UNIVERSAL_COLS]


receipts_norm = normalize(receipts_df, "receipt")
payments_norm = normalize(payments_df, "payment")
bva_norm = normalize(bva_df, "budget_vs_actual", section_col="type")
lgef_norm = normalize(lgef_df, "lgef", section_col="category")
cdf_norm = normalize(cdf_df, "cdf", section_col="category")
revenue_norm = normalize(revenue_detail_df, "revenue_detail", section_col="revenue_category")


combined_df = pd.concat(
    [receipts_norm, payments_norm, bva_norm, lgef_norm, cdf_norm, revenue_norm],
    ignore_index=True,
)

print(f"✅ Combined dataset: {len(combined_df)} rows")
print(f"\nRows by record type:")
print(combined_df["record_type"].value_counts())


# ============================================================
# 13. EXPORT — SINGLE CSV PER YEAR
# ============================================================
out_file = OUTPUT_DIR / (
    f"db-unza26-csc4792-chirundu_{YEAR}_financials.csv"
)
combined_df.to_csv(
    out_file,
    sep="|",
    index=False,
    encoding="utf-8-sig",
)

print(f"\n✅ Saved: {out_file.name}")
print(f"   Rows: {len(combined_df)}")
print(f"   Size: {out_file.stat().st_size / 1024:.1f} KB")

✅ Combined dataset: 129 rows

Rows by record type:
record_type
revenue_detail      62
lgef                18
cdf                 16
budget_vs_actual    14
receipt             11
payment              8
Name: count, dtype: int64

✅ Saved: db-unza26-csc4792-chirundu_2022_financials.csv
   Rows: 129
   Size: 14.7 KB


In [22]:
# ============================================================
# VERIFY OUTPUT FOLDER
# ============================================================
from pathlib import Path

OUTPUT_DIR = Path("finances_output")

print(f"📁 {OUTPUT_DIR.resolve()}")
print("=" * 70)

files = sorted([f for f in OUTPUT_DIR.iterdir() if f.is_file()])
for f in files:
    print(f"  📄 {f.name:<60} {f.stat().st_size / 1024:>8.1f} KB")

print(f"\nTotal files: {len(files)}")

if len(files) == 2:
    print("✅ Correct — one raw OCR + one combined CSV")
else:
    print(f"⚠️  Expected 2 files, found {len(files)}")

📁 C:\Users\USER\Videos\chitundu\finances_output
  📄 chirundu_2022_raw_ocr.csv                                        57.8 KB
  📄 db-unza26-csc4792-chirundu_2022_financials.csv                   14.7 KB

Total files: 2
✅ Correct — one raw OCR + one combined CSV


In [23]:
# ============================================================
# EXPORT — SINGLE UNIFIED CSV (NO DELETION)
# ============================================================
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

UNIVERSAL_COLS = [
    "fiscal_year", "record_type", "section", "line_item",
    "amount_current_zmw", "amount_prior_zmw",
    "approved_budget_zmw", "actual_zmw",
    "variance_zmw", "variance_pct", "is_material_variance",
    "notes", "source_document", "source_page",
]


def normalize(df, record_type, section_col=None, notes_default=""):
    out = pd.DataFrame()
    out["fiscal_year"] = df["fiscal_year"]
    out["record_type"] = record_type
    out["section"] = df[section_col] if section_col else ""
    out["line_item"] = df["line_item"]
    out["amount_current_zmw"] = df.get("amount_current_zmw")
    out["amount_prior_zmw"] = df.get("amount_prior_zmw")
    out["approved_budget_zmw"] = df.get("original_budget_zmw")
    out["actual_zmw"] = df.get("actual_zmw")
    out["variance_zmw"] = df.get("variance_zmw")
    out["variance_pct"] = df.get("variance_pct")
    out["is_material_variance"] = df.get("is_material_variance")
    out["notes"] = notes_default
    out["source_document"] = df["source_document"]
    out["source_page"] = df["source_page"]
    return out[UNIVERSAL_COLS]


# --- Normalize each table ---
receipts_norm = normalize(receipts_df, "receipt")
payments_norm = normalize(payments_df, "payment")
bva_norm = normalize(bva_df, "budget_vs_actual", section_col="type")
lgef_norm = normalize(lgef_df, "lgef", section_col="category")
cdf_norm = normalize(cdf_df, "cdf", section_col="category")
revenue_norm = normalize(revenue_detail_df, "revenue_detail",
                         section_col="revenue_category")

# --- Combine ---
combined_df = pd.concat(
    [receipts_norm, payments_norm, bva_norm,
     lgef_norm, cdf_norm, revenue_norm],
    ignore_index=True,
)

# --- Write this year's CSV (no deletion of other years) ---
out_file = OUTPUT_DIR / (
    f"db-unza26-csc4792-chirundu_{YEAR}_financials.csv"
)
combined_df.to_csv(
    out_file,
    sep="|",
    index=False,
    encoding="utf-8-sig",
)

print(f"✅ Saved: {out_file.name}")
print(f"   Rows: {len(combined_df)}")
print(f"   Size: {out_file.stat().st_size / 1024:.1f} KB")

✅ Saved: db-unza26-csc4792-chirundu_2022_financials.csv
   Rows: 129
   Size: 14.7 KB
